In [1]:
import sys
sys.path.append('../src')

from pathlib import Path
from parse_jams import convert_all
from render_midi import load_config, render_clip, get_vsti_track, render_all
from dataset import make_datasets
from torch.utils.data import DataLoader
from pathlib import Path
from model import TCN
import yaml
import reapy
import soundfile as sf
import torch


In [2]:
repo_root = Path('..').resolve()
config = load_config(repo_root / 'configs' / 'b3_organ.yaml')

## Convert all JAMS to MIDI

In [4]:
convert_all(
    annotation_dir=Path('../data/guitarset/annotation'),
    output_dir=Path('../data/midi')
)

Converted 00_BN1-129-Eb_solo.jams
Converted 00_BN1-147-Gb_comp.jams
Converted 00_BN1-147-Gb_solo.jams
Converted 00_BN2-131-B_comp.jams
Converted 00_BN2-131-B_solo.jams
Converted 00_BN2-166-Ab_comp.jams
Converted 00_BN2-166-Ab_solo.jams
Converted 00_BN3-119-G_comp.jams
Converted 00_BN3-119-G_solo.jams
Converted 00_BN3-154-E_comp.jams
Converted 00_BN3-154-E_solo.jams
Converted 00_Funk1-114-Ab_comp.jams
Converted 00_Funk1-114-Ab_solo.jams
Converted 00_Funk1-97-C_comp.jams
Converted 00_Funk1-97-C_solo.jams
Converted 00_Funk2-108-Eb_comp.jams
Converted 00_Funk2-108-Eb_solo.jams
Converted 00_Funk2-119-G_comp.jams
Converted 00_Funk2-119-G_solo.jams
Converted 00_Funk3-112-C#_comp.jams
Converted 00_Funk3-112-C#_solo.jams
Converted 00_Funk3-98-A_comp.jams
Converted 00_Funk3-98-A_solo.jams
Converted 00_Jazz1-130-D_comp.jams
Converted 00_Jazz1-130-D_solo.jams
Converted 00_Jazz1-200-B_comp.jams
Converted 00_Jazz1-200-B_solo.jams
Converted 00_Jazz2-110-Bb_comp.jams
Converted 00_Jazz2-110-Bb_solo.jam

## Render .wav files of VSTi

In [6]:
project = reapy.Project()
track = get_vsti_track(project)

midi_path = sorted((repo_root / 'data' / 'midi').glob('*.mid'))[0]
output_path = repo_root / config['output_dir'] / (midi_path.stem + '.wav')

render_clip(project, track, midi_path, output_path, config)
print(f'Output exists: {output_path.exists()}')

Output exists: True


In [7]:
from render_midi import render_all

render_all(
    midi_dir=repo_root / 'data' / 'midi',
    config=config,
    repo_root=repo_root
)

Skipping 00_BN1-129-Eb_comp.mid
Rendered 00_BN1-129-Eb_solo.mid
Rendered 00_BN1-147-Gb_comp.mid
Rendered 00_BN1-147-Gb_solo.mid
Rendered 00_BN2-131-B_comp.mid
Rendered 00_BN2-131-B_solo.mid
Rendered 00_BN2-166-Ab_comp.mid
Rendered 00_BN2-166-Ab_solo.mid
Rendered 00_BN3-119-G_comp.mid
Rendered 00_BN3-119-G_solo.mid
Rendered 00_BN3-154-E_comp.mid
Rendered 00_BN3-154-E_solo.mid
Rendered 00_Funk1-114-Ab_comp.mid
Rendered 00_Funk1-114-Ab_solo.mid
Rendered 00_Funk1-97-C_comp.mid
Rendered 00_Funk1-97-C_solo.mid
Rendered 00_Funk2-108-Eb_comp.mid
Rendered 00_Funk2-108-Eb_solo.mid
Rendered 00_Funk2-119-G_comp.mid
Rendered 00_Funk2-119-G_solo.mid
Rendered 00_Funk3-112-C#_comp.mid
Rendered 00_Funk3-112-C#_solo.mid
Rendered 00_Funk3-98-A_comp.mid
Rendered 00_Funk3-98-A_solo.mid
Rendered 00_Jazz1-130-D_comp.mid
Rendered 00_Jazz1-130-D_solo.mid
Rendered 00_Jazz1-200-B_comp.mid
Rendered 00_Jazz1-200-B_solo.mid
Rendered 00_Jazz2-110-Bb_comp.mid
Rendered 00_Jazz2-110-Bb_solo.mid
Rendered 00_Jazz2-187-F#

## Test dataset assembly

In [3]:
repo_root = Path('..') 
config = yaml.safe_load(open(repo_root / 'configs' / 'b3_organ.yaml'))

train_ds, val_ds = make_datasets(
    rendered_dir=repo_root / config['output_dir'],
    guitar_dir=repo_root / 'data' / 'guitarset' / 'audio',
)

print(f"Train windows: {len(train_ds)}, Val windows: {len(val_ds)}")

guitar, rendered = train_ds[0]
print(f"Guitar shape: {guitar.shape}, Rendered shape: {rendered.shape}")

loader = DataLoader(train_ds, batch_size=32, shuffle=True)
batch_guitar, batch_rendered = next(iter(loader))
print(f"Batch shapes: {batch_guitar.shape}, {batch_rendered.shape}")


Train windows: 210999, Val windows: 24645
Guitar shape: torch.Size([1, 4096]), Rendered shape: torch.Size([1, 4096])
Batch shapes: torch.Size([32, 1, 4096]), torch.Size([32, 1, 4096])


In [6]:
# spot check first 3 pairs
from dataset import _build_pairs
pairs = _build_pairs(
    rendered_dir=repo_root / config['output_dir'],
    guitar_dir=repo_root / 'data' / 'guitarset' / 'audio',
)

for guitar_path, rendered_path in pairs[:3]:
    g_info = sf.info(str(guitar_path))
    r_info = sf.info(str(rendered_path))
    print(f"{guitar_path.stem}")
    print(f"  guitar:   {g_info.frames} frames, {g_info.channels}ch, {g_info.samplerate}Hz")
    print(f"  rendered: {r_info.frames} frames, {r_info.channels}ch, {r_info.samplerate}Hz")
    print()


00_BN1-129-Eb_comp_mix
  guitar:   984506 frames, 1ch, 44100Hz
  rendered: 1072732 frames, 2ch, 44100Hz

00_BN1-129-Eb_solo_mix
  guitar:   984506 frames, 1ch, 44100Hz
  rendered: 1072732 frames, 2ch, 44100Hz

00_BN1-147-Gb_comp_mix
  guitar:   863725 frames, 1ch, 44100Hz
  rendered: 951959 frames, 2ch, 44100Hz



In [9]:
model = TCN()
print(f"Receptive field: {model.receptive_field()} samples ({model.receptive_field() / 44100 * 1000:.1f}ms)")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

x = torch.randn(4, 1, 4096)
y = model(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {y.shape}")

Receptive field: 4093 samples (92.8ms)
Parameters: 62,177
Input shape:  torch.Size([4, 1, 4096])
Output shape: torch.Size([4, 1, 4096])


In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce GTX 1050 Ti
